# Bab 18: Sistem Rekomendasi

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab18/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab18")
print("Folder kerja:", os.getcwd())

## Listing 18.1: Kemiripan kosinus dan ramalan berbasis pengguna

`kode/bab18/kosinus_mini.py`

In [ ]:
import numpy as np
import pandas as pd

R = pd.DataFrame([[5, 3, 0, 1, 0, 2],
                  [4, 0, 0, 1, 0, 1],
                  [1, 1, 0, 5, 4, 0],
                  [0, 1, 5, 4, 0, 0],
                  [4, 2, 0, 0, 1, 0]],
                 index=["Ani", "Budi", "Citra", "Dedi", "Eka"],
                 columns=["gula aren", "pandan", "V60", "americano",
                          "espresso", "roti"])
X = R.to_numpy(float)
norma = np.linalg.norm(X, axis=1)
S = X @ X.T / np.outer(norma, norma)
print("kemiripan kosinus antarpengguna:")
print(pd.DataFrame(S, R.index, R.index).round(3).to_string())
u = 1
tetangga = [i for i in np.argsort(-S[u]) if i != u][:2]
print("dua tetangga Budi:", [R.index[i] for i in tetangga])
for j in np.where(X[u] == 0)[0]:
    w = S[u, tetangga]
    pred = (w * X[tetangga, j]).sum() / w.sum()
    print(f"  ramalan Budi untuk {R.columns[j]:9s}: {pred:.2f}")

## Listing 18.2: Faktor laten menu dari SVD terpotong

`kode/bab18/faktor.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab18')
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from siapkan import muat

R, _, menu = muat()
X = np.log1p(R.to_numpy(float))
for k in [1, 2, 3, 5, 10]:
    sv = TruncatedSVD(k, random_state=0).fit(X)
    print(f"k={k:2d}: varians terjelaskan {sv.explained_variance_ratio_.sum():.3f}")
sv = TruncatedSVD(2, random_state=0).fit(X)
Q = sv.components_.T * sv.singular_values_ / np.sqrt(len(X))
tab = pd.DataFrame(Q, index=menu["nama_menu"], columns=["f1", "f2"])
tab["kategori"] = menu["kategori"].to_numpy()
print(tab.sort_values("f2").round(3).to_string())
with open("gambar/bab18/faktor.dat", "w") as f:
    f.write("f1 f2 kat nama\n")
    kat = {"kopi susu": 0, "kopi hitam": 1, "non-kopi": 2, "makanan": 3}
    for n, r in tab.iterrows():
        f.write(f"{r.f1:.4f} {r.f2:.4f} {kat[r.kategori]} "
                f"{{{n}}}\n")

## Listing 18.3: Studi kasus perbandingan metode rekomendasi

`kode/bab18/studi_kasus.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab18')
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from siapkan import konten, muat

R, T, menu = muat()
kode = list(menu.index)
u = R.index.intersection(T.index)
A = R.loc[u].to_numpy(float)
lama = A > 0
baru = (T.loc[u].to_numpy() > 0) & ~lama
ada = baru.sum(1) > 0
X = np.log1p(R.to_numpy(float))
pop = R.sum().to_numpy(float)
print(f"{len(R)} anggota latih (Jan-Sep 2025), {ada.sum()} anggota uji "
      f"memesan menu baru di Okt-Des")


def nilai(S, k=3, baris=None):
    S = S.astype(float)
    S[lama] = -np.inf
    top = np.argsort(-S, 1)[:, :k]
    hit = np.take_along_axis(baru, top, 1)
    m = ada if baris is None else ada & baris
    return dict(presisi=hit[m].sum(1).mean() / k,
                recall=(hit[m].sum(1) / baru[m].sum(1)).mean(),
                hit=(hit[m].sum(1) > 0).mean(),
                pop_rata=pop[top[m]].mean() / pop.mean())


def ternorma(M, sumbu):
    return M / np.maximum(np.linalg.norm(M, axis=sumbu, keepdims=True),
                          1e-12)


skor = {}
skor["popularitas"] = np.tile(pop, (len(u), 1))
C = ternorma(konten(kode), 1)
profil = ternorma(np.log1p(A) @ C, 1)
skor["konten"] = profil @ C.T
Si = ternorma(X, 0).T @ ternorma(X, 0)
np.fill_diagonal(Si, 0)
skor["item-kNN"] = np.log1p(A) @ Si
Su = ternorma(np.log1p(A), 1) @ ternorma(X, 1).T
tet = np.argsort(-Su, 1)[:, 1:51]
skor["user-kNN"] = np.array([(Su[i, tet[i]][:, None] *
                              (X[tet[i]] > 0)).sum(0)
                             for i in range(len(u))])
sv = TruncatedSVD(2, random_state=0).fit(X)
skor["SVD k=2"] = sv.transform(np.log1p(A)) @ sv.components_
skor["gabungan"] = sum(np.argsort(np.argsort(skor[n], 1), 1)
                       for n in ["item-kNN", "SVD k=2", "konten"])

tab = pd.DataFrame({n: nilai(s) for n, s in skor.items()}).T
print("\nsemua anggota uji, top-3:")
print(tab.round(3).to_string())
sedikit = A.sum(1) <= 3
print(f"\nanggota dengan <= 3 pesanan latih ({(sedikit & ada).sum()} orang):")
print(pd.DataFrame({n: nilai(s, baris=sedikit) for n, s in skor.items()})
      .T[["presisi", "hit"]].round(3).to_string())